# The Supercharged Expansion: Auditing Tesla’s Market Penetration Through Infrastructure and Production Scaling

![Alt text](Tesla-prices-2025-lineup.jpg)

The introduction of the Model S created a seismic shift in the EV industry- EVs with niche technological features were now accessible challenging other mainstream established brands, proving EVs can be better than their gas powered countrerpats. This analysis examines Tesla's market penetration in different regions, by way of different models, and the supercharger density. The entry of other models will also be examined, together with factors determining price elasticity, operational efficiency, and supply chain scaling. 

## Business Questions
1. Operational Efficiency
- What is the Delivery-to-Production Ratio across regions, and can we identify specific 'Operational Bottlenecks' where production scaling outpaces logistics capacity?
- To what extent does End-of-Quarter Cyclicality impact delivery efficiency, and has Tesla successfully 'smoothed' this curve between 2015 and 2025?
2. Market penetration
- What is the Price Elasticity of Demand per region, and how does the entry of 'Mass-Market' models (3/Y) shift the correlation between Avg_Price_USD and Market_Share? 
- Does Supercharger Density act as a leading indicator for market penetration, and what is the 'Saturation Point' where additional stations no longer yield proportional delivery growth?
3. Sustainability
- What is the Carbon-Offset Efficiency (CO2 saved per kWh produced), and how has the regional energy mix transition influenced the 'Net-Positive' impact of Tesla's fleet over time?

## Key Performance Indicators


| Strategic Pillar | Primary KPI | Data Columns Involved |
| :--- | :--- | :--- |
| Operations | Delivery Ratio | Production_Units, Estimated_Deliveries |
| Penetration | Infrastructure Lead | Charging_Stations, YearMonth, Region |
| Elasticity | Price-to-Volume | Avg_Price_USD, Estimated_Deliveries, Model |


### Data Understanding

In [199]:
# Importing libraries
import polars as pl
import plotly.express as px

In [200]:
# importing the data from csv 
data = pl.read_csv('tesla_data.csv')

In [201]:
# Explore the general overview of the data, first 5 columns
data.head(5)

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
i64,i64,str,str,i64,i64,f64,i64,i64,f64,str,i64
2023,5,"""Europe""","""Model S""",17646,17922,92874.27,120,704,1863.42,"""Interpolated (Month)""",12207
2015,2,"""Asia""","""Model X""",3797,4164,62205.65,75,438,249.46,"""Official (Quarter)""",7640
2019,1,"""North America""","""Model X""",8411,9189,117887.32,82,480,605.59,"""Interpolated (Month)""",14071
2021,2,"""North America""","""Model 3""",6555,7311,89294.91,120,712,700.07,"""Official (Quarter)""",9333
2016,12,"""Middle East""","""Model Y""",12374,13537,114846.78,120,661,1226.88,"""Estimated (Region)""",8722


In [202]:
data.glimpse()

Rows: 2640
Columns: 12
$ Year                 <i64> 2023, 2015, 2019, 2021, 2016, 2020, 2015, 2020, 2022, 2021
$ Month                <i64> 5, 2, 1, 2, 12, 4, 11, 6, 4, 3
$ Region               <str> 'Europe', 'Asia', 'North America', 'North America', 'Middle East', 'Asia', 'Asia', 'Europe', 'Europe', 'Middle East'
$ Model                <str> 'Model S', 'Model X', 'Model X', 'Model 3', 'Model Y', 'Model X', 'Model 3', 'Cybertruck', 'Model S', 'Model Y'
$ Estimated_Deliveries <i64> 17646, 3797, 8411, 6555, 12374, 4656, 7717, 8410, 15145, 7790
$ Production_Units     <i64> 17922, 4164, 9189, 7311, 13537, 5043, 7976, 9192, 15760, 8208
$ Avg_Price_USD        <f64> 92874.27, 62205.65, 117887.32, 89294.91, 114846.78, 86930.57, 87588.21, 73815.61, 69993.86, 50591.6
$ Battery_Capacity_kWh <i64> 120, 75, 82, 120, 120, 82, 82, 100, 100, 82
$ Range_km             <i64> 704, 438, 480, 712, 661, 477, 475, 592, 563, 485
$ CO2_Saved_tons       <f64> 1863.42, 249.46, 605.59, 700.07, 1226.88, 333.14, 5

In [203]:
data.describe()

statistic,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,f64
"""count""",2640.0,2640.0,"""2640""","""2640""",2640.0,2640.0,2640.0,2640.0,2640.0,2640.0,"""2640""",2640.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0
"""mean""",2020.0,6.5,null,null,9922.199621,10655.847348,84907.34033,87.05947,500.257576,744.076989,null,8932.133712
"""std""",3.162877,3.452707,null,null,3935.950093,4260.600858,20123.258036,20.836265,120.868549,353.221224,null,3469.565883
"""min""",2015.0,1.0,"""Asia""","""Cybertruck""",48.0,50.0,50003.7,60.0,330.0,3.07,"""Estimated (Region)""",3002.0
"""25%""",2017.0,4.0,null,null,7292.0,7829.0,67727.18,75.0,418.0,499.73,null,5899.0
"""50%""",2020.0,7.0,null,null,9860.0,10550.0,85073.32,82.0,470.0,699.65,null,8902.0
"""75%""",2023.0,9.0,null,null,12506.0,13469.0,102371.33,100.0,586.0,943.29,null,11937.0
"""max""",2025.0,12.0,"""North America""","""Model Y""",25704.0,28939.0,119965.36,120.0,719.0,2548.55,"""Official (Quarter)""",14996.0


In [204]:
# Investigating the columns in the data
data.columns

['Year',
 'Month',
 'Region',
 'Model',
 'Estimated_Deliveries',
 'Production_Units',
 'Avg_Price_USD',
 'Battery_Capacity_kWh',
 'Range_km',
 'CO2_Saved_tons',
 'Source_Type',
 'Charging_Stations']

In [205]:
data_lazy = pl.scan_csv("tesla_data.csv")

In [206]:
null_report = data_lazy.null_count().collect()

In [207]:
null_report

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0


In [208]:
# Ensure we have the counts (this works whether 'data' was eager or lazy)
# If eager, just use null_report = data.null_count()
null_report = data.null_count()

# Step-by-step logic to flag issues
print("--- TESLA DATA INTEGRITY AUDIT ---")
has_issues = False

for col in null_report.columns:
    # In Polars, we access the value in the first row [0]
    count = null_report[col][0]
    
    if count > 0:
        print(f"⚠️ AUDIT ALERT: There are {count} nulls on '{col}' column.")
        has_issues = True

if not has_issues:
    print("✅ DATA CLEAN: No nulls detected in any functional columns.")

--- TESLA DATA INTEGRITY AUDIT ---
✅ DATA CLEAN: No nulls detected in any functional columns.


## Data Preparation

In [209]:
# adding a new column for date - month - year
days_in_every_month = {
    1: 31,  # January
    2: 28,  # February (not accounting for leap years)
    3: 31,  # March
    4: 30,  # April
    5: 31,  # May
    6: 30,  # June
    7: 31,  # July
    8: 31,  # August
    9: 30,  # September
    10: 31, # October
    11: 30, # November
    12: 31, # December
}
# add a column for date where format is dd-mm-yyyy with day drawn from days_in_every_month
data = data.with_columns(
    pl.col("Month").map_elements(lambda m: f"{days_in_every_month[m]:02d}").alias("day"),
    pl.col("Month").map_elements(lambda m: f"{m:02d}").alias("month_str"),
    pl.col("Year").cast(pl.Utf8).alias("year_str")
).with_columns(
    pl.concat_str(["day", "month_str", "year_str"], separator="-").alias("date")
).drop(["day", "month_str", "year_str"])

data.head(5)

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations,date
i64,i64,str,str,i64,i64,f64,i64,i64,f64,str,i64,str
2023,5,"""Europe""","""Model S""",17646,17922,92874.27,120,704,1863.42,"""Interpolated (Month)""",12207,"""31-05-2023"""
2015,2,"""Asia""","""Model X""",3797,4164,62205.65,75,438,249.46,"""Official (Quarter)""",7640,"""28-02-2015"""
2019,1,"""North America""","""Model X""",8411,9189,117887.32,82,480,605.59,"""Interpolated (Month)""",14071,"""31-01-2019"""
2021,2,"""North America""","""Model 3""",6555,7311,89294.91,120,712,700.07,"""Official (Quarter)""",9333,"""28-02-2021"""
2016,12,"""Middle East""","""Model Y""",12374,13537,114846.78,120,661,1226.88,"""Estimated (Region)""",8722,"""31-12-2016"""


In [210]:
# adding a new column for delivery ratio
data = data.with_columns(
    (pl.col("Estimated_Deliveries") / pl.col("Production_Units")).alias("Delivery_ratio")
)

# Display the updated DataFrame
data.head(5)

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations,date,Delivery_ratio
i64,i64,str,str,i64,i64,f64,i64,i64,f64,str,i64,str,f64
2023,5,"""Europe""","""Model S""",17646,17922,92874.27,120,704,1863.42,"""Interpolated (Month)""",12207,"""31-05-2023""",0.9846
2015,2,"""Asia""","""Model X""",3797,4164,62205.65,75,438,249.46,"""Official (Quarter)""",7640,"""28-02-2015""",0.911864
2019,1,"""North America""","""Model X""",8411,9189,117887.32,82,480,605.59,"""Interpolated (Month)""",14071,"""31-01-2019""",0.915334
2021,2,"""North America""","""Model 3""",6555,7311,89294.91,120,712,700.07,"""Official (Quarter)""",9333,"""28-02-2021""",0.896594
2016,12,"""Middle East""","""Model Y""",12374,13537,114846.78,120,661,1226.88,"""Estimated (Region)""",8722,"""31-12-2016""",0.914087


In [211]:
# using quantiles to check for outliers for all numeric columns
numeric_cols = data.select(pl.col(pl.Float64)).columns
for col in numeric_cols:
    q1 = data.select(pl.col(col).quantile(0.25)).item()
    q3 = data.select(pl.col(col).quantile(0.75)).item()
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = data.filter((pl.col(col) < lower_bound) | (pl.col(col) > upper_bound))
    
    print(f"Column: {col}")
    print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
    print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
    print(f"Number of Outliers: {outliers.height}\n")

Column: Avg_Price_USD
Q1: 67727.18, Q3: 102371.33, IQR: 34644.15000000001
Lower Bound: 15760.95499999998, Upper Bound: 154337.55500000002
Number of Outliers: 0

Column: CO2_Saved_tons
Q1: 499.73, Q3: 943.29, IQR: 443.55999999999995
Lower Bound: -165.6099999999999, Upper Bound: 1608.6299999999999
Number of Outliers: 45

Column: Delivery_ratio
Q1: 0.8999085086916743, Q3: 0.9641486392184229, IQR: 0.0642401305267486
Lower Bound: 0.8035483129015514, Upper Bound: 1.0605088350085459
Number of Outliers: 0



In [212]:
fig = px.box(data.to_pandas(), y="CO2_Saved_tons", title="Boxplot of CO2 Saved")
fig.show()

In [213]:
# removing outliers from the data
q1 = data.select(pl.col("CO2_Saved_tons").quantile(0.25)).item()
q3 = data.select(pl.col("CO2_Saved_tons").quantile(0.75)).item()
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

data = data.filter((pl.col("CO2_Saved_tons") >= lower_bound) & (pl.col("CO2_Saved_tons") <= upper_bound))

# checking dimensions of the data after removing outliers
data.describe()


statistic,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations,date,Delivery_ratio
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,f64,str,f64
"""count""",2595.0,2595.0,"""2595""","""2595""",2595.0,2595.0,2595.0,2595.0,2595.0,2595.0,"""2595""",2595.0,"""2595""",2595.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,"""0""",0.0
"""mean""",2020.006551,6.492871,null,null,9783.313295,10505.233911,84848.606998,86.534489,497.26474,725.675846,null,8945.389981,null,0.932694
"""std""",3.163307,3.450309,null,null,3816.499258,4129.575002,20092.228158,20.608362,119.617211,326.306486,null,3468.573407,null,0.037281
"""min""",2015.0,1.0,"""Asia""","""Cybertruck""",48.0,50.0,50003.7,60.0,330.0,3.07,"""Estimated (Region)""",3002.0,"""28-02-2015""",0.869683
"""25%""",2017.0,4.0,null,null,7202.0,7794.0,67717.11,75.0,418.0,495.66,null,5902.0,null,0.899912
"""50%""",2020.0,6.0,null,null,9776.0,10449.0,85034.64,82.0,468.0,692.7,null,8915.0,null,0.93224
"""75%""",2023.0,9.0,null,null,12351.0,13305.0,102342.21,100.0,583.0,932.58,null,11946.0,null,0.964398
"""max""",2025.0,12.0,"""North America""","""Model Y""",25410.0,28802.0,119965.36,120.0,719.0,1604.82,"""Official (Quarter)""",14996.0,"""31-12-2025""",1.0


In [214]:
# The "Zombie Hunt"
categorical_cols = ["Region", "Model", "Source_Type"]

for col in categorical_cols:
    unique_vals = data[col].unique().to_list()
    print(f"--- {col} ({len(unique_vals)} unique) ---")
    print(sorted(unique_vals))

--- Region (4 unique) ---
['Asia', 'Europe', 'Middle East', 'North America']
--- Model (5 unique) ---
['Cybertruck', 'Model 3', 'Model S', 'Model X', 'Model Y']
--- Source_Type (3 unique) ---
['Estimated (Region)', 'Interpolated (Month)', 'Official (Quarter)']


In [215]:
# Detective work: Find all impossible Cybertruck entries
impossible_trucks = data.filter(
    (pl.col("Model") == "Cybertruck") & (pl.col("Year") < 2023)
)

print(f"Number of impossible Cybertruck rows: {impossible_trucks.height}")
print(impossible_trucks.select(["Year", "Month", "Region", "Estimated_Deliveries"]))

Number of impossible Cybertruck rows: 380
shape: (380, 4)
┌──────┬───────┬───────────────┬──────────────────────┐
│ Year ┆ Month ┆ Region        ┆ Estimated_Deliveries │
│ ---  ┆ ---   ┆ ---           ┆ ---                  │
│ i64  ┆ i64   ┆ str           ┆ i64                  │
╞══════╪═══════╪═══════════════╪══════════════════════╡
│ 2020 ┆ 6     ┆ Europe        ┆ 8410                 │
│ 2020 ┆ 4     ┆ North America ┆ 4408                 │
│ 2016 ┆ 12    ┆ North America ┆ 5224                 │
│ 2018 ┆ 12    ┆ North America ┆ 8427                 │
│ 2015 ┆ 3     ┆ North America ┆ 5716                 │
│ …    ┆ …     ┆ …             ┆ …                    │
│ 2016 ┆ 12    ┆ Asia          ┆ 2907                 │
│ 2016 ┆ 11    ┆ Middle East   ┆ 11120                │
│ 2019 ┆ 12    ┆ North America ┆ 6711                 │
│ 2018 ┆ 3     ┆ Europe        ┆ 13104                │
│ 2020 ┆ 5     ┆ Asia          ┆ 3471                 │
└──────┴───────┴───────────────┴──────────────

In [216]:
# Cybertrucks first appeared in 2023, so we can filter out any rows where Model is "Cybertruck" and Year is less than 2023
data = data.filter(~((pl.col("Model") == "Cybertruck") & (pl.col("Year") < 2023)))
#checking dimensions of the data after removing impossible cybertruck entries
data.describe()

statistic,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations,date,Delivery_ratio
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,f64,str,f64
"""count""",2215.0,2215.0,"""2215""","""2215""",2215.0,2215.0,2215.0,2215.0,2215.0,2215.0,"""2215""",2215.0,"""2215""",2215.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,"""0""",0.0
"""mean""",2020.266817,6.490293,null,null,9836.56614,10562.327765,85001.761914,86.562077,497.362528,730.091314,null,8979.36298,null,0.932737
"""std""",3.218153,3.449176,null,null,3785.048513,4099.718299,19881.165278,20.527431,119.102083,325.848391,null,3468.697383,null,0.037558
"""min""",2015.0,1.0,"""Asia""","""Cybertruck""",121.0,139.0,50003.7,60.0,330.0,8.66,"""Estimated (Region)""",3002.0,"""28-02-2015""",0.869712
"""25%""",2017.0,4.0,null,null,7319.0,7866.0,68273.4,75.0,419.0,502.12,null,5909.0,null,0.899674
"""50%""",2020.0,6.0,null,null,9804.0,10487.0,85522.74,82.0,469.0,695.92,null,8942.0,null,0.93224
"""75%""",2023.0,9.0,null,null,12364.0,13318.0,101984.52,100.0,583.0,932.58,null,11987.0,null,0.965069
"""max""",2025.0,12.0,"""North America""","""Model Y""",25410.0,28802.0,119965.36,120.0,719.0,1604.82,"""Official (Quarter)""",14996.0,"""31-12-2025""",1.0


In [217]:
# are deliveries just a formula of production units?
interpolated_only = data.filter(pl.col("Source_Type") == "Interpolated (Month)")
correlation = interpolated_only.select(
    pl.corr("Estimated_Deliveries", "CO2_Saved_tons")
)
print(f"Correlation in Interpolated Data: {correlation[0,0]}")

Correlation in Interpolated Data: 0.8249245062957276


When we get to the ML Phase, we should train the model without the interpolated rows first to see the "Pure" signals, then add them back to see if they improve the "Decade Long" trend.

## Exploratory Data Analysis

In [218]:
# Calculate delivery to production ratio by region and year
delivery_production_ratio_by_region_year = data.group_by("Region", "Year").agg(
    (pl.col("Estimated_Deliveries").sum() / pl.col("Production_Units").sum()).alias("Delivery_to_Production_Ratio")
).sort("Year")

# Plot multiple line graph of delivery to production ratio across regions across years
fig = px.line(
    delivery_production_ratio_by_region_year.to_pandas(), 
    x="Year", 
    y="Delivery_to_Production_Ratio", 
    color="Region", 
    title="Delivery to Production Ratio by Region Across Years",
    markers=True
)
fig.show()

The mean ratios are virtually similar across all regions, mathematically sterile. However the a multiline graph show a decline in production across regions in 2020 consistent with COVID 19 pandemic disruption. 

In [219]:
# Multi-Line Time Series by Quarter
# Add Quarter column
data = data.with_columns(
    (((pl.col("Month") - 1) // 3) + 1).alias("Quarter")
)

# Create Year-Quarter string for plotting
plot_data = data.with_columns(
    pl.concat_str(["Year", "Quarter"], separator="-Q").alias("Year-Quarter")
)

# Group by Year-Quarter and Region, calculate mean delivery ratio
quarterly_data = plot_data.group_by("Year-Quarter", "Region").agg(
    pl.mean("Delivery_ratio").alias("Avg_Delivery_Ratio")
).sort("Year-Quarter")

# Plot multi-line time series
fig = px.line(
    quarterly_data.to_pandas(),
    x="Year-Quarter",
    y="Avg_Delivery_Ratio",
    color="Region",
    title="Delivery Ratio Time Series by Quarter and Region",
    markers=True
)
fig.update_xaxes(type='category')  # Treat as categorical for proper ordering
fig.show()

In [220]:
# 1. Aggregate to Yearly to remove monthly 'noise'
yearly_elasticity = data_clean.group_by(["Year", "Region"]).agg([
    pl.col("Avg_Price_USD").mean(),
    pl.col("Global_Share_Proxy").mean(),
    pl.col("Production_Units").sum()
]).sort("Year")

# 2. The Trajectory Plot
fig = px.line(
    yearly_elasticity.to_pandas(), 
    x="Avg_Price_USD", 
    y="Global_Share_Proxy", 
    color="Region",
    markers=True,
    text="Year", # Label the years so we see the direction
    title="Tesla's Price Elasticity Trajectory: The Pivot to Mass Market (2015-2025)",
    labels={"Global_Share_Proxy": "Market Share %", "Avg_Price_USD": "Avg Selling Price ($)"}
)

# Reverse the X-axis because Price Drops are the 'Growth' lever
fig.update_xaxes(autorange="reversed") 
fig.show()

The Tesla Decade (2015–2025): From Luxury Moat to Commodity War
Between 2015 and 2025, Tesla underwent a fundamental Seismic Shift in its operational and economic DNA. The data reveals a company that successfully solved its physical logistics crisis only to be met by a new, more ruthless challenge: Market Saturation and Price Inelasticity.

#### 1. The Logistics Triumph: Smoothing the "Wave"
The most visible success in the audit is the "smoothing" of the delivery curve.

- The Era of Chaos (2015–2020): The quarterly data exposed a "Logistics Wave" where Tesla was a startup in a constant state of emergency. The 2018 Asia Collapse and 2020 COVID Shock were the low points of this era, characterized by high production but "lumpy," inefficient deliveries.

- The Era of Stability (2021–2025): By establishing regional Gigafactories (Shanghai, Berlin, Texas), Tesla transformed its logistics from a single global "pipe" into a distributed network. As of 2025, the delivery ratio has converged across major regions, proving that Tesla can now move 2 million units with more stability than it moved 200,000 in 2017.

#### 2. The Price-Volume Paradox: The Death of the Luxury Moat
The Price Elasticity Trajectory unmasked the high cost of this growth.

- Luxury Era (Pre-2022): Tesla operated as a Veblen Good. High prices and high market share co-existed because the product was a "status revolution."

- Commodity Era (2023–2025): The entry of the Model 3 and Y into a hyper-competitive market (the "BYD Effect") destroyed pricing power. The vertical drop in the 2023 data proves that Price Elasticity has flipped: Tesla is now forced to cut prices just to defend a shrinking share. In North America, we see a "Reverse Drift"—prices are higher in 2025 due to inflation/mix, but market share is lower than the 2016 peak.

#### 3. Regional Divergence: The "New Frontiers"
- The Middle East Anomaly: This remains the "Growth Beacon." While Asia and Europe are flattening, the Middle East (UAE/Saudi Arabia) showed a Price-Share climb in 2022-2023, driven by new market entry, high oil prices, and the "Luxury Status" that the West has already lost.

- The 2025 Infrastructure Moat: Infrastructure is no longer a "growth driver" in mature markets like the US (where it's now expected); it has become a Defensive Moat. In Asia, however, the 49% increase in charging stations remains a leading indicator of the next volume spike.

In [221]:
# 1. Find the Leading Indicator: Test 6-month Lag
data_lagged = data_clean.with_columns([
    pl.col("Charging_Stations").shift(6).over("Region").alias("Stations_Lead_6M")
])

# 2. Visualize Saturation
# We use a Log-Log or Semi-Log plot to see the 'Flattening' clearly
fig = px.scatter(
    data_lagged.to_pandas(),
    x="Charging_Stations", 
    y="Estimated_Deliveries",
    color="Region",
    trendline="lowess", # Shows the non-linear 'wobble' and flattening
    title="Infrastructure Saturation Audit: At what point do Plugs stop selling Cars?",
    labels={"Charging_Stations": "Total Supercharger Count", "Estimated_Deliveries": "Monthly Sales Volume"}
)
fig.show()

In [222]:
# 1. Independent Time Series (Decoupling Check)
# Aggregating globally to see the big picture
global_trends = data_clean.group_by("Year").agg([
    pl.col("Charging_Stations").mean().alias("Avg_Stations"),
    pl.col("Estimated_Deliveries").sum().alias("Total_Deliveries")
]).sort("Year")

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=global_trends["Year"], y=global_trends["Avg_Stations"], name="Global Superchargers"), secondary_y=False)
fig.add_trace(go.Scatter(x=global_trends["Year"], y=global_trends["Total_Deliveries"], name="Global Deliveries"), secondary_y=True)

fig.update_layout(title_text="The Decoupling: Infrastructure vs. Sales Volume (2015-2025)")
fig.show()

# 2. Regional Efficiency Check (2025 Snapshot)
latest_year = data_clean.filter(pl.col("Year") == 2025).group_by("Region").agg([
    (pl.col("Estimated_Deliveries").sum() / pl.col("Charging_Stations").mean()).alias("Cars_per_Station")
])

fig2 = px.bar(latest_year.to_pandas(), x="Region", y="Cars_per_Station", 
              title="Infrastructure Efficiency: Cars per Supercharger (2025)")
fig2.show()

In [223]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. THE SURGERY
df_clean = data.filter(
    ~((pl.col("Model") == "Cybertruck") & (pl.col("Year") < 2023))
)

# 2. AGGREGATION & CUMULATION
yearly_stats = (
    df_clean.group_by(["Year", "Region"])
    .agg([
        pl.col("Production_Units").sum().alias("Yearly_Units"),
        pl.col("Charging_Stations").sum().alias("Yearly_Stations")
    ])
    .sort(["Region", "Year"])
    .with_columns([
        pl.col("Yearly_Units").cum_sum().over("Region").alias("Total_Fleet"),
        pl.col("Yearly_Stations").cum_sum().over("Region").alias("Total_Stations")
    ])
)

# 3. THE VISUALIZATION
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("<b>Cumulative Vehicle Fleet (Regional Market Size)</b>", 
                    "<b>Cumulative Supercharger Network (Regional Infrastructure Moat)</b>")
)

# Professional "Tableau" Colors
colors = px.colors.qualitative.T10 
regions = sorted(yearly_stats["Region"].unique().to_list())

for i, region in enumerate(regions):
    reg_df = yearly_stats.filter(pl.col("Region") == region)
    
    # Trace for Total Fleet (Stock)
    fig.add_trace(
        go.Scatter(x=reg_df["Year"], y=reg_df["Total_Fleet"],
                   name=f"{region} (Fleet)", mode='lines+markers',
                   line=dict(color=colors[i % len(colors)], width=3),
                   legendgroup=region),
        row=1, col=1
    )
    
    # Trace for Total Stations (Stock)
    fig.add_trace(
        go.Scatter(x=reg_df["Year"], y=reg_df["Total_Stations"],
                   name=f"{region} (Stations)", mode='lines+markers',
                   line=dict(color=colors[i % len(colors)], width=3, dash='dash'),
                   legendgroup=region, showlegend=False),
        row=2, col=1
    )

# 4. UPDATED LAYOUT (Legend on the Right)
fig.update_layout(
    height=900,
    title_text="<b>Regional Stock Audit: The Cumulative Infrastructure Moat (2015-2025)</b>",
    template="plotly_white",
    # Legend Position: Right-Side
    legend=dict(
        orientation="v",       # Vertical
        yanchor="middle",      # Center vertically
        y=0.5,                 # Middle of the plot
        xanchor="left",        # Anchor the left of the legend
        x=1.05,                # Push it slightly outside the plot area
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="Black",
        borderwidth=1
    ),
    margin=dict(r=150) # Ensure there's enough room on the right for the legend
)

fig.update_yaxes(title_text="Total Vehicles", row=1, col=1)
fig.update_yaxes(title_text="Total Stations", row=2, col=1)
fig.update_xaxes(title_text="Fiscal Year", row=2, col=1, tickmode='linear')

fig.show()

In [225]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. THE SURGERY & CUMULATIVE AGGREGATION
df_clean = data.filter(
    ~((pl.col("Model") == "Cybertruck") & (pl.col("Year") < 2023))
)

yearly_stats = (
    df_clean.group_by(["Year", "Region"])
    .agg([
        pl.col("Production_Units").sum().alias("Yearly_Units"),
        pl.col("Charging_Stations").sum().alias("Yearly_Stations")
    ])
    .sort(["Region", "Year"])
    .with_columns([
        pl.col("Yearly_Units").cum_sum().over("Region").alias("Total_Fleet"),
        pl.col("Yearly_Stations").cum_sum().over("Region").alias("Total_Stations")
    ])
)

# 2. BUILDING THE COMBO PLOT (One Subplot per Region)
regions = sorted(yearly_stats["Region"].unique().to_list())
fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f"<b>{r} Fleet Stock vs. Infrastructure Moat</b>" for r in regions],
    specs=[[{"secondary_y": True}] for _ in regions]
)

# Professional "Tableau" Colors for Consistency
colors = px.colors.qualitative.T10

for i, region in enumerate(regions):
    reg_df = yearly_stats.filter(pl.col("Region") == region)
    region_color = colors[i % len(colors)]
    
    # BARS: Cumulative Fleet (Primary Y-Axis)
    # Using the region_color with some transparency for readability
    fig.add_trace(
        go.Bar(
            x=reg_df["Year"], 
            y=reg_df["Total_Fleet"],
            name=f"{region} Fleet (Bars)",
            marker_color=region_color,
            opacity=0.6,
            legendgroup=region
        ),
        row=i+1, col=1, secondary_y=False
    )
    
    # LINE: Cumulative Stations (Secondary Y-Axis)
    # Using the EXACT SAME region_color, but making it a solid line
    fig.add_trace(
        go.Scatter(
            x=reg_df["Year"], 
            y=reg_df["Total_Stations"],
            name=f"{region} Stations (Line)",
            mode='lines+markers',
            # THIS IS THE FIX: same color, thick, solid line
            line=dict(color=region_color, width=4), 
            legendgroup=region
        ),
        row=i+1, col=1, secondary_y=True
    )

# 3. LAYOUT & LABELS
fig.update_layout(
    height=320 * len(regions),
    title_text="<b>Continental Stock Audit (2015-2025): Total Vehicles (Bars) vs. Total Stations (Line)</b>",
    template="plotly_white",
    hovermode="x unified",
    # Legend Position: Right-Side, Vertical
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.05,
        bgcolor="rgba(255, 255, 255, 0.7)",
        bordercolor="Black",
        borderwidth=1
    ),
    margin=dict(r=180) # Adjust right margin for the larger right-hand legend
)

# Individual Axis Labels and Professional Formatting
for i in range(len(regions)):
    fig.update_yaxes(title_text="<b>Fleet Size</b> (Cars)", row=i+1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="<b>Total Stations</b> (Plugs)", row=i+1, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Audit Year", row=i+1, col=1, tickmode='linear')

fig.show()

In [226]:
# 1. DATA PREP & RATIO CALCULATION
# Using the existing df_clean logic from previous turns
df_clean = data.filter(
    ~((pl.col("Model") == "Cybertruck") & (pl.col("Year") < 2023))
)

# Convert to Pandas for visualization if needed, but keeping Polars logic for speed
audit_data = (
    df_clean.group_by(["Year", "Region"])
    .agg([
        pl.col("Production_Units").sum().alias("Yearly_Units"),
        pl.col("Charging_Stations").sum().alias("Yearly_Stations")
    ])
    .sort(["Region", "Year"])
    .with_columns([
        pl.col("Yearly_Units").cum_sum().over("Region").alias("Total_Fleet"),
        pl.col("Yearly_Stations").cum_sum().over("Region").alias("Total_Stations")
    ])
    .with_columns([
        # Calculate Stations per 1000 Cars (The Margin)
        (pl.col("Total_Stations") / (pl.col("Total_Fleet") / 1000)).alias("Margin_Ratio")
    ])
)

# 2. BUILDING THE MARGIN AUDIT
regions = sorted(audit_data["Region"].unique().to_list())
fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f"<b>{r}: Fleet Pressure vs. Infrastructure Margin</b>" for r in regions],
    specs=[[{"secondary_y": True}] for _ in regions]
)

colors = px.colors.qualitative.T10

for i, region in enumerate(regions):
    reg_df = audit_data.filter(pl.col("Region") == region)
    region_color = colors[i % len(colors)]
    
    # BARS: Fleet Size (The Pressure)
    fig.add_trace(
        go.Bar(
            x=reg_df["Year"], y=reg_df["Total_Fleet"],
            name=f"{region} Fleet (Pressure)",
            marker_color=region_color, opacity=0.4,
            legendgroup=region
        ),
        row=i+1, col=1, secondary_y=False
    )
    
    # LINE: Margin Ratio (The Deficiency Indicator)
    fig.add_trace(
        go.Scatter(
            x=reg_df["Year"], y=reg_df["Margin_Ratio"],
            name=f"{region} Margin (Plugs/1k Cars)",
            mode='lines+markers',
            line=dict(color=region_color, width=4),
            legendgroup=region
        ),
        row=i+1, col=1, secondary_y=True
    )

# 3. LAYOUT & LABELS
fig.update_layout(
    height=350 * len(regions),
    title_text="<b>Continental Deficiency Audit: Is Infrastructure Keeping Pace with Fleet Growth?</b>",
    template="plotly_white",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.05, borderwidth=1),
    margin=dict(r=200)
)

for i in range(len(regions)):
    fig.update_yaxes(title_text="Fleet Size", row=i+1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Stations per 1k Cars", row=i+1, col=1, secondary_y=True)

fig.show()